# Free Wan 2.2 Video Generation API
Run on **free Google Colab T4 GPU**.

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**
3. Copy the ngrok URL → paste into your `.env` as `COLAB_API_URL`

> First time? You need:
> - **ngrok token** (free): https://dashboard.ngrok.com/get-started/your-authtoken
> - **HF token** (free, for fast downloads): https://huggingface.co/settings/tokens

In [ ]:
# Cell 0: Clean old 14B cache (run ONCE if switching to 5B)
# import shutil
# shutil.rmtree('/content/drive/MyDrive/wan_cache', ignore_errors=True)
# print('Deleted 40GB cache')

In [ ]:
# Cell 1: Setup tokens + Drive
import os

# ── HF TOKEN (for fast model downloads) ──
# Get your free token: https://huggingface.co/settings/tokens
# Then uncomment the next lines and add your token

# ── NGROK TOKEN (for public API URL) ──
NGROK_TOKEN = input('Paste your ngrok authtoken: ') or os.environ.get('NGROK_AUTHTOKEN', '')
if NGROK_TOKEN:
    get_ipython().system(f'ngrok config add-authtoken {NGROK_TOKEN}')
else:
    print('WARNING: No ngrok token. Get one at https://dashboard.ngrok.com/get-started/your-authtoken')

# ── Mount Drive for model cache ──
from google.colab import drive
drive.mount('/content/drive')
CACHE_DIR = '/content/drive/MyDrive/wan_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = CACHE_DIR
os.environ['HF_HUB_CACHE'] = CACHE_DIR
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'
print('\n✅ Drive mounted, cache:', CACHE_DIR)

In [ ]:
# Cell 2: Free space + install deps
!rm -rf /content/sample_data /root/.cache/pip
!pip install -q torch diffusers transformers accelerate pillow flask flask-cors pyngrok
!pip install -q git+https://github.com/huggingface/diffusers.git
print('\n✅ Dependencies installed')

In [ ]:
# Cell 3: Load Wan model (5B = 15GB, fits Colab T4)
import os, torch, base64, gc
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
torch.cuda.empty_cache()
gc.collect()

from diffusers import WanPipeline
from diffusers.utils import export_to_video
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"

print(f'Loading {MODEL_ID} (this takes ~3 min)...')
pipe = WanPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()
print('\n✅ Model loaded!')

In [ ]:
# Cell 4: API server
app = Flask(__name__)
CORS(app)

@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'model': 'Wan 2.2 5B'})

@app.route('/generate', methods=['POST'])
def generate():
    data = request.json
    prompt = data.get('prompt', '')
    if not prompt:
        return jsonify({'error': 'prompt required'}), 400
    num_frames = data.get('num_frames', 40)
    width = 848 if data.get('aspect_ratio') == '9:16' else 1280
    height = 1280 if data.get('aspect_ratio') == '9:16' else 720
    gc.collect()
    torch.cuda.empty_cache()
    frames = pipe(prompt, num_frames=min(num_frames, 40), width=width, height=height, num_inference_steps=25).frames[0]
    path = '/tmp/wan_output.mp4'
    export_to_video(frames, path, fps=8)
    del frames
    with open(path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    return jsonify({'videos': [{'url': f'data:video/mp4;base64,{b64}'}], 'provider': 'colab-wan'})

public_url = ngrok.connect(5000).public_url
print('='*60)
print(f'API URL: {public_url}')
print('='*60)
print(f'\nAdd to .env:\nCOLAB_API_URL={public_url}')
app.run(port=5000, debug=False)